# Семинар 7–8. Метрики качества и интерпретируемость ML-моделей

**Дисциплина «ИИ и машинное обучение»** · НИУ ВШЭ, Высшая школа бизнеса  
Магистратура «Бизнес-аналитика и цифровые решения в финансах»

| Часть | О чём |
|---|---|
| 1. Метрики качества моделей и их связь с бизнес-результатом | Отказы оборудования, банковские звонки и распределение обращений по трём категориям - выбор метрики под задачу |
| 2. Диагностика и интерпретируемость ML-моделей | Собственная реализация LIME для текста, SHAP для текста и Deep SHAP для нейросети, распознающей цифры |

**Как работать.** Выполняйте ячейки сверху вниз. В ячейках с заголовком
«Задание» замените пропуск своим кодом; самопроверка находится в конце
той же ячейки.

Данные об оборудовании — [AI4I 2020 Predictive Maintenance](https://archive.ics.uci.edu/dataset/601/ai4i+2020+predictive+maintenance+da)
(синтетические данные UCI); банковские данные —
[Bank Marketing](https://archive.ics.uci.edu/dataset/222/bank+marketing) (UCI).
Обе подготовленные таблицы находятся в `data/` репозитория курса.
Текстовые сообщения составлены для этого семинара. Изображения цифр —
[Digits](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_digits.html)
из `scikit-learn`.

### Установка сторонних библиотек

Перед запуском семинара нужны библиотеки для LIME и SHAP:

```python
%pip install -q lime==0.2.0.1 shap==0.49.1
```

Следующая ячейка выполняет ту же установку автоматически **только в
Colab**. Остальные используемые библиотеки (`numpy`, `pandas`,
`matplotlib`, `scikit-learn`, `torch`) входят в обычное окружение Colab.

In [ ]:
## ── установка дополнений в Colab ──────────────────────────────────
import sys
import subprocess
if "google.colab" in sys.modules:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "lime==0.2.0.1", "shap==0.49.1"], check=True)

### Импорт библиотек и вспомогательные функции

In [ ]:
## ── библиотеки ─────────────────────────────────────────────────────
import sys
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
    confusion_matrix, roc_auc_score, average_precision_score, auc,
    roc_curve, precision_recall_curve, precision_recall_fscore_support)
from sklearn.datasets import load_digits

def check(ok, hint):
    if not ok:
        raise AssertionError(f"✗ {hint}")
    print(f"✓ верно")

def load_course_csv(name):
    """Локальная копия при наличии, иначе опубликованный файл курса."""
    for path in (Path("data") / name, Path("../data") / name,
                 Path("../../ml_ai_course_gsb_hse/data") / name):
        if path.exists():
            return pd.read_csv(path)
    url = f"https://raw.githubusercontent.com/Rimest/ml_ai_course_gsb_hse/main/data/{name}"
    return pd.read_csv(url)

print(f"Готово. scikit-learn {__import__('sklearn').__version__}")

# Часть 1. Метрики качества моделей и их связь с бизнес-результатом

## Отказы оборудования: accuracy не отражает истинное качество модели

**Положительный класс** — отказ станка. Для решения о проверке доступны
измерения до отказа: температура, обороты, момент и износ. Столбцы с
причиной отказа исключены: причина устанавливается после события.
Деление на обучение и проверку фиксировано.

`RandomForestClassifier` строит ансамбль деревьев; `n_estimators` задаёт
число деревьев, `min_samples_leaf` — минимальный размер листа,
`class_weight="balanced"` увеличивает вес редкого класса.

In [ ]:
machine = load_course_csv("machine_failures.csv")
machine_features = ["температура_цеха", "температура_станка", "обороты",
                    "момент", "износ_резца"]
machine_X = machine[machine_features]
machine_y = machine["отказ"].astype(int)
machine_X_train, machine_X_test, machine_y_train, machine_y_test = train_test_split(
    machine_X, machine_y, test_size=0.30, stratify=machine_y, random_state=42)
machine_model = RandomForestClassifier(n_estimators=120, min_samples_leaf=5,
    class_weight="balanced", random_state=42, n_jobs=-1)
machine_model.fit(machine_X_train, machine_y_train)
machine_score = machine_model.predict_proba(machine_X_test)[:, 1]
machine_baseline = np.zeros(len(machine_y_test), dtype=int)
print(f"Проверка: {len(machine_y_test)} станков, отказов: {machine_y_test.sum()}")
machine_X

### Задание 1. Найдите четыре исхода решения о проверке станка

Определите **TP**, **FP**, **TN** и **FN** для правила «всегда исправен» и
для решений случайного леса при пороге 0,5. Сохраните значения в `machine_counts`
как словарь с ключами `Всегда исправен` и `Случайный лес`; каждому ключу
соответствует кортеж `(TP, FP, TN, FN)`.

Для каждого объекта сравните истинный класс `machine_y_test` с прогнозом.
`machine_baseline` уже содержит прогноз постоянного правила, а
`machine_pred` — прогноз леса.

In [ ]:
machine_pred = (machine_score >= 0.5).astype(int)
machine_true = machine_y_test.to_numpy()
machine_counts = {}
for name, pred in (("Всегда исправен", machine_baseline),
                   ("Случайный лес", machine_pred)):
    # TODO
    raise NotImplementedError("замените эту строку своим решением")
    machine_counts[name] = (tp, fp, tn, fn)
check(set(machine_counts) == {"Всегда исправен", "Случайный лес"}
      and all(tuple(machine_counts[name]) == tuple(map(int, confusion_matrix(
          machine_true, pred).ravel()[[3, 1, 0, 2]]))
          for name, pred in (("Всегда исправен", machine_baseline),
                             ("Случайный лес", machine_pred))),
      "проверьте TP, FP, TN и FN для обоих правил")

После подсчёта четырёх исходов получаем три метрики:

$$\operatorname{precision}=\frac{TP}{TP+FP},\qquad
\operatorname{recall}=\frac{TP}{TP+FN},\qquad
\operatorname{accuracy}=\frac{TP+TN}{TP+FP+TN+FN}.$$

**Precision** показывает долю реальных отказов среди станков, которые
модель предложила проверить. 

**Recall** показывает долю найденных отказов
среди всех действительно отказавших станков. 

**Accuracy** показывает долю
верных решений среди всех станков. Если модель никого не отправила на
проверку, знаменатель precision равен нулю; ниже в таком случае
используется значение 0.

In [ ]:
## ── метрики precision, recall, accuracy ────────────────────────────
def metrics_from_counts(counts):
    tp, fp, tn, fn = counts
    n = tp + fp + tn + fn
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    accuracy = (tp + tn) / n
    return accuracy, precision, recall

for name, counts in machine_counts.items():
    accuracy, precision, recall = metrics_from_counts(counts)
    tp, fp, tn, fn = counts
    print(f"{name:<18} TP={tp:>3}  FP={fp:>3}  TN={tn:>4}  FN={fn:>3}  "
          f"accuracy={accuracy:.3f}  precision={precision:.3f}  recall={recall:.3f}")

Те же метрики можно вычислить готовыми функциями `scikit-learn`:
`accuracy_score`, `precision_score`, `recall_score`. Функция
`confusion_matrix` возвращает значения в порядке **TN, FP, FN, TP**.
Сравните её результат с собственным подсчётом:

In [ ]:
for name, pred in (("Всегда исправен", machine_baseline),
                   ("Случайный лес", machine_pred)):
    tn, fp, fn, tp = confusion_matrix(machine_true, pred).ravel()
    print(f"{name:<18} accuracy={accuracy_score(machine_true, pred):.3f}  "
          f"precision={precision_score(machine_true, pred, zero_division=0):.3f}  "
          f"recall={recall_score(machine_true, pred):.3f}  "
          f"FN={fn}")

Accuracy равняется доле верных ответов модели как для положительного, так
и для отрицательного класса. Если отказов мало, правильные ответы по
исправным станкам (отрицательный класс) могут почти полностью определять
accuracy. Для решения о проверке станка нужно отдельно смотреть на
пропущенные отказы (FN) и ложные тревоги (FP).

## Порог: качество ранжирования и стоимость решения

ROC-кривая показывает зависимость TPR (доли найденных отказов) от FPR
(доли ложных тревог среди исправных). PR-кривая показывает precision
(долю отказов среди отправленных на проверку) при разной полноте.
**AP — способ рассчитать площадь под PR-кривой**:

$$AP=\sum_{i=1}^{m}(R_i-R_{i-1})P_i,$$

где $R_i$ — recall после очередного шага, $P_i$ — соответствующий
precision. В `precision_recall_curve` точки возвращаются в обратном
порядке: recall убывает. Поэтому для применения формулы массивы ниже
развёрнуты.

### Задание 2. Рассчитайте AP по точкам PR-кривой

Используйте готовые массивы `machine_recall_up` и `machine_precision_up`,
полученные функцией `precision_recall_curve` и упорядоченные по возрастанию
recall. Вычислите `machine_ap_manual` по формуле выше: для каждого шага
умножьте прирост recall на precision в конце шага и сложите результаты.
Стоимость пропущенного отказа (FN) — 20 условных единиц, ложной тревоги
(FP) — 1 условная единица. Расчёт потерь при разных порогах и графики
подготовлены. По графику определите порог с минимальными потерями.

In [ ]:
### Предварительный подсчет machine_recall_up, machine_precision_up
machine_thresholds = np.arange(0.05, 0.91, 0.05)
machine_precision_curve, machine_recall_curve, _ = precision_recall_curve(
    machine_y_test, machine_score)

machine_recall_up = machine_recall_curve[::-1]
machine_precision_up = machine_precision_curve[::-1]
## ── вычислите AP по точкам PR-кривой ───────────────────────────────
# TODO
raise NotImplementedError("замените эту строку своим решением")

check(abs(machine_ap_manual - average_precision_score(
      machine_y_test, machine_score)) < 1e-10, "проверьте сумму площадей")

machine_roc_auc = roc_auc_score(machine_y_test, machine_score)
machine_costs = []
for threshold in machine_thresholds:
    pred = (machine_score >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(machine_y_test, pred).ravel()
    machine_costs.append(20 * fn + fp)  # стоимость пропусков и ложных тревог
machine_ap_trapezoid = float(auc(machine_recall_up, machine_precision_up))
machine_best_idx = int(np.argmin(machine_costs))
machine_best_threshold = float(machine_thresholds[machine_best_idx])
machine_best_loss = int(machine_costs[machine_best_idx])
machine_fpr, machine_tpr, _ = roc_curve(machine_y_test, machine_score)
fig, axes = plt.subplots(1, 3, figsize=(14, 3.8))
axes[0].plot(machine_fpr, machine_tpr, label=f"ROC-AUC={machine_roc_auc:.3f}")
axes[0].plot([0, 1], [0, 1], "--", color="gray")
axes[0].set(xlabel="FPR", ylabel="TPR", title="Отказы: ROC")
axes[0].legend()
axes[1].plot(machine_recall_curve, machine_precision_curve,
             label=f"AP={machine_ap_manual:.3f}")
axes[1].axhline(machine_y_test.mean(), linestyle="--", color="gray")
axes[1].set(xlabel="Recall", ylabel="Precision", title="Отказы: PR")
axes[1].legend()
axes[2].plot(machine_thresholds, machine_costs, marker="o")
axes[2].scatter([machine_best_threshold], [machine_best_loss],
                color="crimson", s=85, zorder=3)
axes[2].set(xlabel="Порог", ylabel="Потери, усл. ед.",
            title="Потери при разных порогах")
plt.tight_layout()
plt.show()
print(f"AP по точкам PR-кривой: {machine_ap_manual:.4f}; "
      f"площадь по трапециям: {machine_ap_trapezoid:.4f}")
print(f"Лучший порог на проверочной сетке: {machine_best_threshold:.2f}; "
      f"потери: {machine_best_loss} условных единиц")

Готовая функция `average_precision_score(y_true, scores)` вычисляет AP
без ручного суммирования точек.

## Банковские звонки

Положительный класс — клиент открыл вклад. Используем только поля,
доступные до текущего звонка. Длительность звонка исключена: до звонка
она неизвестна. Обучаем логистическую регрессию и случайный лес, затем сравниваем
качество списка из 200 клиентов на проверочной выборке.

`make_pipeline(StandardScaler(), LogisticRegression(...))` нормирует
числовые признаки перед линейной моделью. `RandomForestClassifier`
строит ансамбль деревьев.

In [ ]:
bank = load_course_csv("bank_marketing.csv")
bank_features = ["возраст", "баланс", "звонков_в_кампании",
                 "дней_с_прошлой_кампании", "звонков_раньше"]
bank_X = bank[bank_features]
bank_y = bank["открыл_вклад"].astype(int)
bank_X_train, bank_X_test, bank_y_train, bank_y_test = train_test_split(
    bank_X, bank_y, test_size=0.25, stratify=bank_y, random_state=42)
bank_linear = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
bank_forest = RandomForestClassifier(n_estimators=120, min_samples_leaf=20,
                                     random_state=42, n_jobs=-1)
bank_linear.fit(bank_X_train, bank_y_train)
bank_forest.fit(bank_X_train, bank_y_train)
bank_linear_score = bank_linear.predict_proba(bank_X_test)[:, 1]
bank_forest_score = bank_forest.predict_proba(bank_X_test)[:, 1]
bank_k = 200
print(f"Проверка: {len(bank_y_test)} клиентов; лимит: {bank_k} звонков")
bank_X

## Сколько полезных контактов среди top-K клиентов

Модель упорядочивает клиентов по оценке вероятности открытия вклада.
Можно позвонить только первым $K$ людям в списке. Пусть $TP@K$ - количество клиентов, которые действительно открыли вклад, тогда можно ввести метрики:

$$\operatorname{precision@K}=\frac{TP@K}{K},\qquad
\operatorname{recall@K}=\frac{TP@K}{N_+},$$

где $N_+$ — **все** открывшие вклад в проверочной выборке. Знаменатели
разные: precision@K отвечает на вопрос «насколько полезен список из K
звонков?», recall@K — «какую долю всех будущих вкладчиков охватили?».
На рисунке зелёные карточки обозначают открывших вклад. Среди первых
пяти — два таких клиента, а во всём списке их четыре: precision@5 = 2/5,
recall@5 = 2/4.

In [ ]:
## ── рисунок: два разных знаменателя ───────────────────────────────
example_bank_labels = np.array([1, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0])
fig, ax = plt.subplots(figsize=(10, 2.8))
for i, label in enumerate(example_bank_labels):
    color = "#2e8b57" if label else "#e0e7ef"
    ax.scatter(i + 1, 0, s=650, marker="s", color=color, edgecolor="#42566d")
    ax.text(i + 1, 0, str(i + 1), ha="center", va="center", fontsize=10)
ax.axvline(5.5, color="#d84a3a", linewidth=2, linestyle="--")
ax.text(3, 0.65, "Первые K = 5: 2 вкладчика / 5 звонков", ha="center")
ax.text(8.7, -0.65, "Во всём списке: 4 вкладчика", ha="center")
ax.set(xlim=(0, 13), ylim=(-1, 1), title="Клиенты по убыванию оценки модели")
ax.axis("off")
plt.show()

### Задание 3. Реализуйте precision@K и recall@K

Для каждого из двух наборов оценок `bank_linear_score` и
`bank_forest_score` отсортируйте клиентов по **убыванию** оценки и найдите
число открывших вклад среди первых `bank_k = 200`. Вычислите
`precision_at_k` и `recall_at_k` по формулам выше.

In [ ]:
bank_true = bank_y_test.to_numpy()
bank_results = []
for name, score in (("Линейная", bank_linear_score), ("Лес", bank_forest_score)):
    # TODO
    raise NotImplementedError("замените эту строку своим решением")
    check(selected_positive == int(bank_true[
          np.argsort(-score)[:bank_k]].sum())
          and np.isclose(precision_at_k, selected_positive / bank_k)
          and np.isclose(recall_at_k, selected_positive / bank_true.sum()),
          "проверьте порядок клиентов и знаменатели метрик")
    bank_results.append((name, roc_auc_score(bank_true, score),
                         average_precision_score(bank_true, score),
                         precision_at_k, recall_at_k, selected_positive))
for name, roc, ap, precision_k, recall_k, deposits in bank_results:
    print(f"{name:<10} ROC-AUC={roc:.3f}  AP={ap:.3f}  "
          f"precision@{bank_k}={precision_k:.3f}  recall@{bank_k}={recall_k:.3f}  "
          f"вкладов={deposits}")
bank_ks = np.arange(50, 1001, 50)
for name, score in (("Линейная", bank_linear_score), ("Лес", bank_forest_score)):
    order = np.argsort(-score)
    hits = np.cumsum(bank_true[order])
    plt.plot(bank_ks, hits[bank_ks - 1] / bank_ks,
             label=name)
plt.axvline(bank_k, linestyle=":", color="crimson", label=f"Лимит {bank_k} звонков")
plt.xlabel("K — сколько первых клиентов обзвонить")
plt.ylabel("precision@K — доля открывших вклад среди первых K")
plt.title("Как меняется полезность списка при росте числа звонков")
plt.legend()
plt.show()

Каждая точка графика показывает **precision@K** при указанном числе
звонков $K$: сколько из первых $K$ клиентов действительно открыли вклад,
делённое на $K$. Вертикальная линия отмечает доступный лимит в 200
звонков. Сравнивать модели для этой кампании нужно у этой линии.

## Распределение обращений: micro и macro при трёх классах

Служба поддержки относит обращения к трём классам: обычное, срочное,
критическое. Обычных обращений гораздо больше. Для каждого класса
можно считать его положительным, а остальные — отрицательными, получая
собственные TP, FP и FN. **Macro** сначала рассчитывает precision,
recall и F1 для каждого класса, затем берёт обычное среднее трёх
значений. **Micro** сначала складывает TP, FP и FN всех классов,
затем один раз применяет формулы метрик:

$$P_{micro}=\frac{\sum_c TP_c}{\sum_c(TP_c+FP_c)},\quad
R_{micro}=\frac{\sum_c TP_c}{\sum_c(TP_c+FN_c)},\quad
F_{1,micro}=\frac{2P_{micro}R_{micro}}{P_{micro}+R_{micro}}.$$

Для macro усредняют **готовые значения метрик**.
На рисунке слева строки показывают истинные классы обращений, столбцы — прогнозы. Диагональ
содержит верные ответы. Справа показана полнота для **тех же трёх
классов**: доля найденных обращений каждого типа. Синяя линия — средняя
полнота классов (macro), красная — доля найденных обращений среди всех
100 обращений (micro).

In [ ]:
triage_names = ["Обычное", "Срочное", "Критическое"]
triage_true = np.array([0] * 90 + [1] * 8 + [2] * 2)
triage_pred = np.array([0] * 86 + [1] * 3 + [2] + [0] * 3 + [1] * 5 + [0] * 2)
triage_matrix = confusion_matrix(triage_true, triage_pred, labels=[0, 1, 2])
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].imshow(triage_matrix, cmap="Blues")
axes[0].set(xticks=range(3), yticks=range(3), xticklabels=triage_names,
            yticklabels=triage_names, xlabel="ПРОГНОЗ", ylabel="ИСТИНА",
            title="Распределение 100 обращений")
plt.setp(axes[0].get_xticklabels(), rotation=20, ha="right")
for i in range(3):
    for j in range(3):
        axes[0].text(j, i, str(triage_matrix[i, j]), ha="center", va="center",
                     color="white" if triage_matrix[i, j] > 35 else "#17334d")
triage_tp_example = np.diag(triage_matrix)
triage_total_example = triage_matrix.sum(axis=1)
triage_recall_example = triage_tp_example / triage_total_example
triage_macro_recall_example = float(triage_recall_example.mean())
triage_micro_recall_example = float(triage_tp_example.sum() / triage_matrix.sum())
axes[1].barh(triage_names[::-1], triage_recall_example[::-1], color="#78a9dc")
axes[1].axvline(triage_macro_recall_example, color="#0057b8", linewidth=2,
                label=f"macro-recall = {triage_macro_recall_example:.3f}")
axes[1].axvline(triage_micro_recall_example, color="#c43c39", linewidth=2,
                label=f"micro-recall = {triage_micro_recall_example:.3f}")
for y, (tp, total) in enumerate(zip(triage_tp_example[::-1], triage_total_example[::-1])):
    axes[1].text(0.02, y, f"{tp}/{total}", va="center", color="#17334d")
axes[1].set(xlim=(0, 1.05), xlabel="Доля найденных обращений",
            title="Recall для каждого типа обращения")
axes[1].legend(loc="lower center", bbox_to_anchor=(0.5, -0.40))
fig.tight_layout()
plt.show()

## Как micro и macro оценивают распределение обращений

Из 90 обычных обращений модель распознала 86, из 8 срочных — 5, а из
2 критических — ни одного. Поэтому полнота по классам равна
$86/90\approx0{,}956$, $5/8=0{,}625$ и $0/2=0$.
**Macro-recall** — среднее этих трёх долей: около $0{,}527$.
**Micro-recall** сначала складывает найденные обращения и общее число
обращений: $(86+5+0)/(90+8+2)=0{,}91$. Высокое micro-значение здесь
получается за счёт многочисленных обычных обращений. Если особенно важны
критические обращения, дополнительно рассматривайте recall именно этого
класса: его значение равно нулю.

Ниже те же расчёты выполнены для precision, recall и F1; таблица
позволяет сравнить результаты по каждому типу обращений.

In [ ]:
triage_counts = np.zeros((3, 3), dtype=int)
for class_id in range(3):
    tp = np.sum((triage_true == class_id) & (triage_pred == class_id))
    fp = np.sum((triage_true != class_id) & (triage_pred == class_id))
    fn = np.sum((triage_true == class_id) & (triage_pred != class_id))
    triage_counts[class_id] = (tp, fp, fn)

check(np.array_equal(triage_counts[:, 0], np.diag(triage_matrix)),
      "проверьте TP каждого класса")
triage_tp, triage_fp, triage_fn = triage_counts.T
triage_p = np.divide(triage_tp, triage_tp + triage_fp,
                     out=np.zeros(3, dtype=float), where=(triage_tp + triage_fp) > 0)
triage_r = np.divide(triage_tp, triage_tp + triage_fn,
                     out=np.zeros(3, dtype=float), where=(triage_tp + triage_fn) > 0)
triage_f1 = np.divide(2 * triage_p * triage_r, triage_p + triage_r,
                      out=np.zeros(3, dtype=float), where=(triage_p + triage_r) > 0)
micro_p = triage_tp.sum() / (triage_tp.sum() + triage_fp.sum())
micro_r = triage_tp.sum() / (triage_tp.sum() + triage_fn.sum())
micro_f1 = 2 * micro_p * micro_r / (micro_p + micro_r)
macro_p = triage_p.mean()
macro_r = triage_r.mean()
macro_f1 = triage_f1.mean()
triage_metrics = (micro_p, micro_r, micro_f1, macro_p, macro_r, macro_f1)
sk_p, sk_r, sk_f, _ = precision_recall_fscore_support(
    triage_true, triage_pred, average="macro", zero_division=0)
check(np.allclose(triage_metrics[3:], (sk_p, sk_r, sk_f))
      and np.isclose(triage_metrics[0], accuracy_score(triage_true, triage_pred)),
      "проверьте порядок усреднения")
for name, tp, fp, fn, p, r, f1 in zip(
        triage_names, triage_tp, triage_fp, triage_fn, triage_p, triage_r, triage_f1):
    print(f"{name:<15} TP={tp:>2}  FP={fp:>2}  FN={fn:>2}  "
          f"precision={p:.3f}  recall={r:.3f}  F1={f1:.3f}")
print(f"micro: precision={micro_p:.3f}, recall={micro_r:.3f}, F1={micro_f1:.3f}")
print(f"macro: precision={macro_p:.3f}, recall={macro_r:.3f}, F1={macro_f1:.3f}")

# Часть 2. Диагностика и интерпретируемость ML-моделей

## Текстовые сообщения: модель и способ объяснения

Рассмотрим обычные сообщения и сообщения со спамом. Повторяемые
шаблоны позволяют понять, как модель принимает решение. Кодирование
TF–IDF (term frequency – inverse document frequency, см. [wiki](https://ru.wikipedia.org/wiki/TF-IDF)) переводит слова
в числовые признаки; логистическая регрессия оценивает класс по ним.

In [ ]:
normal_core = [
    "Позвоните мне после встречи", "Давайте встретимся на вокзале",
    "Я отправлю отчёт завтра", "Ужин готов дома",
    "Ваша запись подтверждена на понедельник", "Спасибо за помощь сегодня",
    "Поезд прибывает в шесть", "Принесите пожалуйста документы",
    "Наше занятие начинается в девять", "Я отправил фотографии",
]
spam_core = [
    "Заберите бесплатный приз сейчас", "Выиграйте денежную награду сегодня",
    "Акция заканчивается переходите по ссылке", "Вы выиграли купон ответьте сейчас",
    "Получите бесплатный подарок по звонку", "Эксклюзивный приз ждёт вас",
    "Срочная акция заберите денежную награду", "Перейдите по ссылке и выиграйте бонус",
    "Специальная скидка доступна сегодня", "Ответьте сейчас и получите приз",
]
normal_tail = ["До скорой встречи", "Сообщите мне пожалуйста", "Благодарю вас", "Поговорим позже"]
spam_tail = ["Акция скоро закончится", "Перейдите по ссылке", "Ответьте немедленно", "Успейте сегодня"]
text_messages = []
text_labels = []
for i in range(200):
    text_messages.append(f"{normal_core[i % 10]}. {normal_tail[(i // 10) % 4]}.")
    text_labels.append(0)
    text_messages.append(f"{spam_core[i % 10]}. {spam_tail[(i // 10) % 4]}.")
    text_labels.append(1)
text_train, text_test, text_y_train, text_y_test = train_test_split(
    text_messages, text_labels, test_size=0.25, stratify=text_labels, random_state=42)
text_model = make_pipeline(TfidfVectorizer(ngram_range=(1, 2)),
                           LogisticRegression(max_iter=1000, random_state=42))

In [ ]:
for message, label in list(zip(text_train, text_y_train))[:4]:
    print(f"Класс {label}: {message}")

In [ ]:
text_model.fit(text_train, text_y_train)
text_score = text_model.predict_proba(text_test)[:, 1]
text_case = "Заберите бесплатный приз прямо сейчас. Перейдите по ссылке и получите подарок."
text_case_score = float(text_model.predict_proba([text_case])[0, 1])
print(f"Корпус: {len(text_messages)} сообщений.\n")
print(f"Сообщение: \n``{text_case}``\n")
print(f"Вероятность рекламного сообщения: {text_case_score:.3f}")

## LIME: объясняем решение модели для конкретного объекта

LIME (Local Interpretable Model-agnostic Explanations) создаёт близкие
варианты объекта, получает ответы исходной модели и обучает на них
простую взвешенную модель. Для текста варианты создаются скрытием
отдельных слов. Поскольку само скрытие меняет текст, объяснение
зависит от того, как устроена маска, по которому происходило скрытие слов.

Обозначения: `Z[i, j]=1`, если слово `j` оставлено в варианте `i`;
`scores[i]` — ответ исходной модели; `weights[i]` — вес варианта по
близости к исходному сообщению. `Ridge` — линейная регрессия с
регуляризацией; `alpha` определяет её силу. `sample_weight` передаёт
веса вариантов. Последняя строка должна содержать все слова.

Для $m$ слов возможны $2^m$ масок. Мы берём случайную часть масок.
Локальная модель $g(z)=\beta_0+\sum_{j=1}^{m}\beta_j z_j$
обучается взвешенной линейной регрессией:

$$\hat\beta=\arg\min_{\beta_0,\beta}
\left[\sum_{i=1}^{N}w_i\bigl(f(z_i)-\beta_0-
\sum_{j=1}^{m}\beta_j z_{ij}\bigr)^2
+\alpha\sum_{j=1}^{m}\beta_j^2\right],$$

где $f(z_i)$ — вероятность рекламного класса по исходной модели,
$w_i=\exp[-(d_i/0{,}5)^2]$, $d_i$ — доля скрытых слов, а
$\alpha=0{,}01$ — коэффициент регуляризации в `Ridge`. Чем ближе
вариант к исходному сообщению, тем больше его вес в обучении.

In [ ]:
lime_distances = np.linspace(0, 1, 201)
lime_weights = np.exp(-(lime_distances / 0.5) ** 2)
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(lime_distances, lime_weights, linewidth=2, color="#0057b8")
for distance in (0.0, 0.25, 0.5, 1.0):
    weight = float(np.exp(-(distance / 0.5) ** 2))
    ax.scatter(distance, weight, color="#c43c39", zorder=3)
    ax.annotate(f"d={distance:g}; w={weight:.2f}", (distance, weight),
                xytext=(-88, 10) if distance == 1 else (6, 8),
                textcoords="offset points",
                fontsize=9)
ax.set(xlim=(-0.02, 1.12), ylim=(-0.02, 1.12),
       xlabel="d — доля скрытых слов", ylabel="w — вес варианта",
       title="LIME: близкие варианты сильнее влияют на локальную модель")
ax.grid(alpha=0.25)
fig.tight_layout()
plt.show()

На графике исходный текст имеет расстояние $d=0$ и вес $w=1$.
Если скрыть половину слов, $d=0{,}5$ и вес снижается примерно до $0{,}37$.
Вариант без всех слов почти не влияет на локальную линейную модель:
при $d=1$ его вес около $0{,}02$.

### Задание 6. Реализуйте LIME для конкретного сообщения

Создайте 400 случайных двоичных масок слов в `text_masks` и добавьте
последнюю строку из единиц для исходного сообщения. Готовый код
построит по маскам тексты `text_variants`. Вычислите долю скрытых слов
`text_distance`, получите вероятности рекламного класса
`text_variant_scores` от `text_model` для всех вариантов и рассчитайте
`text_weights` по формуле выше. Подготовленную линейную модель
`text_lime_model` готовый код обучит с этими весами. Сравните её прогноз
с исходной моделью и назовите слова с наибольшим положительным и
отрицательным вкладом.

In [ ]:
text_words = text_case.split()
text_rng = np.random.default_rng(42)
print(f"Сообщение: {text_case}")
print(f"Слов: {len(text_words)}; возможных масок: {2 ** len(text_words)}")
# TODO
raise NotImplementedError("замените эту строку своим решением")
text_variants = [" ".join(w for w, keep in zip(text_words, row) if keep)
                 for row in text_masks]
# TODO
raise NotImplementedError("замените эту строку своим решением")
text_lime_model = Ridge(alpha=0.01)
text_lime_model.fit(text_masks, text_variant_scores,
                    sample_weight=text_weights)
text_lime_coef = text_lime_model.coef_
check(text_masks.shape == (401, len(text_words)) and
      (text_masks[-1] == 1).all(), "проверьте маски слов")
check(len(text_variants) == 401 and len(text_lime_coef) == len(text_words),
      "проверьте варианты и коэффициенты")
check(abs(text_variant_scores[-1] - text_case_score) < 1e-10,
      "последний вариант должен быть исходным сообщением")
text_lime_pred = float(text_lime_model.predict(text_masks[-1:])[0])
print(f"Исходная модель: {text_case_score:.3f}; локальная: {text_lime_pred:.3f}")
for variant, weight in zip(text_variants[:3], text_weights[:3]):
    print(f"Вариант: {variant:<55} вес={weight:.3f}")
for word, effect in sorted(zip(text_words, text_lime_coef),
                           key=lambda pair: -abs(pair[1])):
    print(f"{word:<15} коэффициент={effect:+.3f}")
plt.barh(text_words[::-1], text_lime_coef[::-1])
plt.axvline(0, color="black", linewidth=0.8)
plt.xlabel("Коэффициент локальной модели")
plt.title("LIME: вклад сохранения слова в окрестности")
plt.show()

## Готовая реализация LIME для того же сообщения

Библиотека `lime` предоставляет `LimeTextExplainer`. Мы передаём ему
то же сообщение `text_case` и метод `predict_proba` нашей модели.
Explainer сам создаёт изменённые тексты, получает для них прогнозы
и обучает локальную интерпретируемую модель. `class_names` задаёт
названия двух выходов, `random_state` фиксирует случайные варианты,
`num_samples=400` задаёт число созданных текстов, а `num_features`
ограничивает число слов, показанных в объяснении.

`as_list(label=1)` возвращает слова и их локальные веса для рекламного
класса; `as_pyplot_figure(label=1)` рисует эти веса. Положительный вес
повышает оценку рекламного класса в локальном приближении. Числа могут
отличаться от нашей реализации: библиотека использует собственные
разбиение текста, набор изменённых сообщений и выбор признаков.

In [ ]:
try:
    from lime.lime_text import LimeTextExplainer
except ImportError as exc:
    raise RuntimeError("Для локального запуска установите lime в отдельное окружение") from exc

ready_lime = LimeTextExplainer(
    class_names=["обычное", "рекламное"], random_state=42)
ready_lime_explanation = ready_lime.explain_instance(
    text_case, text_model.predict_proba, labels=(1,),
    num_features=len(text_words), num_samples=400)
print(f"Вероятность рекламного класса: "
      f"{ready_lime_explanation.predict_proba[1]:.3f}")
for word, weight in ready_lime_explanation.as_list(label=1):
    print(f"{word:<15} вес LIME={weight:+.3f}")
ready_lime_figure = ready_lime_explanation.as_pyplot_figure(label=1)
ready_lime_figure.axes[0].set_title("LIME: важность слов для рекламного класса")
ready_lime_figure.axes[0].set_xlabel("Локальный вес слова")
plt.tight_layout()
plt.show()

## Deep SHAP: нейросеть распознаёт рукописные цифры

`load_digits` содержит изображения цифр 8 × 8 пикселей. Небольшая
нейросеть обучается на этих изображениях. `DeepExplainer` приближает
значения SHAP для нейросети относительно фоновых изображений. Сначала
каждое изображение разворачивается в 64 числа. Первый линейный слой
переводит их в 32 промежуточных значения, `ReLU` заменяет отрицательные
значения нулями, а второй линейный слой выдаёт 10 **логитов** — по одному
на класс цифры. Наибольший логит определяет прогноз.

Deep SHAP использует правила обратного распределения вклада по слоям
(семейство DeepLIFT) и усредняет результат по фоновым изображениям.
Для выбранного класса $c$ и фона $B$ получаем приближённое разложение:

$$f_c(x)\approx \frac{1}{|B|}\sum_{b\in B}f_c(b)
+\sum_{j=1}^{64}\phi^{\mathrm{Deep}}_{j,c}(x;B).$$

Первое слагаемое — средний логит класса на фоне; $\phi_{j,c}$ — вклад
пикселя $j$ относительно этого фона. Метод не перебирает все
$2^{64}$ подмножеств пикселей. Меняя фон, мы меняем точку отсчёта
объяснения. Ниже сначала посмотрим входные изображения и прогнозы сети,
а затем карту вкладов для одного изображения.

In [ ]:
try:
    import torch
    from torch import nn
except ImportError as exc:
    raise RuntimeError("Для локального запуска установите torch в отдельное окружение") from exc

digits = load_digits()
digits_X = digits.data.astype(np.float32) / 16.0
digits_y = digits.target.astype(np.int64)
digits_X_train, digits_X_test, digits_y_train, digits_y_test = train_test_split(
    digits_X, digits_y, test_size=0.2, stratify=digits_y, random_state=42)
torch.manual_seed(42)
digits_net = nn.Sequential(nn.Linear(64, 32), nn.ReLU(), nn.Linear(32, 10))
digits_optimizer = torch.optim.Adam(digits_net.parameters(), lr=0.01)
digits_train_tensor = torch.tensor(digits_X_train)
digits_train_labels = torch.tensor(digits_y_train)
for epoch in range(35):
    digits_optimizer.zero_grad()
    digits_logits = digits_net(digits_train_tensor)
    digits_loss = nn.functional.cross_entropy(digits_logits, digits_train_labels)
    digits_loss.backward()
    digits_optimizer.step()
digits_net.eval()
with torch.no_grad():
    digits_test_logits = digits_net(torch.tensor(digits_X_test))
digits_pred = digits_test_logits.argmax(dim=1).numpy()
print(f"Точность нейросети на проверке: "
      f"{accuracy_score(digits_y_test, digits_pred):.3f}")
fig, axes = plt.subplots(2, 5, figsize=(9, 5))
for i, ax in enumerate(axes.ravel()):
    ax.imshow(digits_X_test[i].reshape(8, 8), cmap="gray_r")
    ax.set_title(f"Истина {digits_y_test[i]} · прогноз {digits_pred[i]}", fontsize=9)
    ax.axis("off")
plt.suptitle("Нейросеть распознаёт цифры на изображениях 8 × 8")
fig.tight_layout(h_pad=2.0, rect=(0, 0, 1, 0.94))
plt.show()
digits_probs = torch.softmax(digits_test_logits[0], dim=0).numpy()
print(f"Первое изображение: прогноз {digits_pred[0]}, "
      f"вероятность класса {digits_pred[0]} = {digits_probs[digits_pred[0]]:.3f}")

### Фоновые изображения: относительно чего объясняется цифра

Выберем первое проверочное изображение и 40 изображений из обучающей
выборки как **фон**. Для каждого класса сеть выдаёт логит. `DeepExplainer` объясняет именно логит
выбранного класса. Базовое значение — средний логит этого класса на
40 фоновых изображениях. Усреднённая картинка справа помогает увидеть,
с какими изображениями сравнивается выбранная цифра; она сама не
является дополнительным объектом для распознавания.

In [ ]:
digits_background = torch.tensor(digits_X_train[:40])
digits_input = torch.tensor(digits_X_test[:1])
digits_class = int(digits_pred[0])
with torch.no_grad():
    digits_background_logits = digits_net(digits_background).numpy()
digits_background_mean_logit = float(digits_background_logits[:, digits_class].mean())
print(f"Выбранный класс: {digits_class}; фоновых изображений: {len(digits_background)}")
print(f"Средний логит класса {digits_class} на фоне: "
      f"{digits_background_mean_logit:.3f}")
fig, axes = plt.subplots(1, 2, figsize=(6.5, 3))
axes[0].imshow(digits_X_test[0].reshape(8, 8), cmap="gray_r")
axes[0].set_title(f"Объясняем прогноз: {digits_class}")
axes[1].imshow(digits_background.numpy().mean(axis=0).reshape(8, 8),
               cmap="gray_r", vmin=0, vmax=1)
axes[1].set_title("Среднее 40 фоновых изображений")
for ax in axes:
    ax.axis("off")
fig.tight_layout()
plt.show()

### `shap.DeepExplainer`: модель, фон и значения SHAP

`shap.DeepExplainer(digits_net, digits_background)` получает обученную
нейросеть и фон. Его `expected_value` содержит базовый логит **для
каждого из 10 классов**. Метод `shap_values(digits_input)` возвращает
вклады пикселей в выходы сети. Здесь вход имеет форму `(1, 64)`:
одно изображение и 64 значения яркости. В используемой версии SHAP
результат имеет форму `(1, 64, 10)` — объект, пиксель, класс. В других
версиях результат может быть списком из 10 массивов; код ниже
обрабатывает оба представления. Параметр `check_additivity=False`
отключает автоматическую проверку библиотеки; равенство базы и суммы
вкладов с логитом мы проверяем и показываем отдельно.

In [ ]:
digits_explainer = shap.DeepExplainer(digits_net, digits_background)
digits_shap_values = digits_explainer.shap_values(
    digits_input, check_additivity=False)
print(f"Форма входа: {tuple(digits_input.shape)}")
if isinstance(digits_shap_values, list):
    print(f"SHAP вернул список из {len(digits_shap_values)} массивов; "
          f"форма одного: {np.asarray(digits_shap_values[0]).shape}")
else:
    print(f"Форма значений SHAP: {np.asarray(digits_shap_values).shape}")

### `expected_value` и `shap_values`: как читать вклад пикселя

Выберем вклады только для предсказанного класса. Его базовый логит
равен `expected_value[digits_class]`. Каждый $\phi_{j,c}$ показывает,
как пиксель $j$ изменил логит класса $c$ относительно фона: положительный
вклад повышает логит, отрицательный снижает. Суммируем 64 вклада и
сравниваем результат с логитом сети для исходного изображения:

$$\text{логит класса}\approx\text{базовый логит}+\sum_{j=1}^{64}\phi_{j,c}.$$

Небольшое расхождение возможно, поскольку Deep SHAP приближает вклады.

In [ ]:
digits_phi_all = np.asarray(digits_shap_values)
if isinstance(digits_shap_values, list):
    digits_phi = np.asarray(digits_shap_values[digits_class])[0]
else:
    digits_phi = digits_phi_all[0, :, digits_class]
digits_attribution = digits_phi.reshape(8, 8)
digits_logit = float(digits_test_logits[0, digits_class])
digits_base = float(np.asarray(digits_explainer.expected_value)[digits_class])
digits_reconstructed_logit = digits_base + float(digits_attribution.sum())
print(f"Класс: {digits_class}; базовый логит: {digits_base:.3f}; "
      f"сумма вкладов: {digits_attribution.sum():+.3f}; "
      f"логит сети: {digits_logit:.3f}")

### Карта вкладов: где сеть нашла признаки выбранного класса

Вернём 64 вклада к расположению пикселей 8 × 8. На карте **красный**
цвет соответствует положительному вкладу в логит выбранного класса,
**синий** — отрицательному. Более насыщенный цвет означает больший
модуль вклада. Карта объясняет прогноз относительно выбранных 40 фоновых
изображений; при другом фоне изменятся базовое значение и вклады.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(7, 3))
axes[0].imshow(digits_X_test[0].reshape(8, 8), cmap="gray_r")
axes[0].set_title(f"Цифра: истинная {digits_y_test[0]}, прогноз {digits_class}")
limit = np.max(np.abs(digits_attribution))
image = axes[1].imshow(digits_attribution, cmap="RdBu_r", vmin=-limit, vmax=limit)
axes[1].set_title("Deep SHAP: вклад пикселей в логит")
for ax in axes:
    ax.axis("off")
fig.colorbar(image, ax=axes[1], fraction=0.046, pad=0.04, label="Вклад в логит")
fig.tight_layout()
plt.show()
positive_pixel = np.unravel_index(np.argmax(digits_attribution), (8, 8))
negative_pixel = np.unravel_index(np.argmin(digits_attribution), (8, 8))
print(f"Наибольший положительный вклад: пиксель {positive_pixel}, "
      f"{digits_attribution[positive_pixel]:+.3f}")
print(f"Наибольший отрицательный вклад: пиксель {negative_pixel}, "
      f"{digits_attribution[negative_pixel]:+.3f}")

# Итоги

| Что прошли |
|---|
| Accuracy, recall, precision, ROC-AUC и AP |
| Метрики для ранжирования precision@K и recall@K |
| Micro и macro метрики многоклассовой классификации |
| Метод LIME для интерпретации решения модели |
| Deep SHAP для интерпретации решения нейронной сети|

**Что дальше.** Нейронные сети!

Источники методов: [метрики scikit-learn](https://scikit-learn.org/stable/modules/model_evaluation.html),
[LIME — Ribeiro et al. (2016)](https://arxiv.org/abs/1602.04938),
[SHAP — Lundberg и Lee (2017)](https://arxiv.org/abs/1705.07874),
[DeepExplainer](https://shap.readthedocs.io/en/stable/generated/shap.DeepExplainer.html).